In [12]:
"""
Generic Tunisian apartment-listing scraper.

Takes a search-results page URL (e.g. a Mubawab or Tayara "apartments for
sale" page) and extracts structured listing data into a pandas DataFrame / CSV.

WORKS BOTH AS A SCRIPT AND IN A JUPYTER NOTEBOOK.
--------------------------------------------------
As a script:
    python scrape_apartments.py "<listing-page-url>" --pages 5 --out data.csv

In a notebook (recommended for this project -- see run_scrape() at the bottom):
    from scrape_apartments import run_scrape
    df = run_scrape("<listing-page-url>", max_pages=5, out_path="data.csv")
    df.head()

HOW TO ADAPT THIS TO A SPECIFIC SITE
-------------------------------------
Real estate sites don't expose a public API, so we have to parse HTML -- and
every site's HTML is different. Every site-specific detail lives in the
SELECTORS dict below. To point this at a real site:

1. Open the listing page in your browser (e.g. Chrome).
2. Right-click a single listing card -> "Inspect".
3. Find the repeating container element for one listing card (e.g. a <div
   class="listingBox">, <article class="listing-card">, etc.) -- that's
   SELECTORS["card"].
4. Inside that card, find the CSS selector for each field (price, surface,
   rooms, location, link) relative to the card. Fill them into SELECTORS.
5. Run it (script or notebook, as above).

If the site renders listings via JavaScript (view page source shows no
listing data, only a loading skeleton), swap requests+BeautifulSoup for
Playwright -- see fetch_with_playwright() near the bottom.
"""

import argparse
import csv
import re
import sys
import time
from dataclasses import asdict, dataclass
from typing import Optional
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

# --------------------------------------------------------------------------
# SITE-SPECIFIC CONFIG -- edit this block per target site
# --------------------------------------------------------------------------
SELECTORS = {
    # Container for a single listing card, scoped from the page/search results
    "card": "div.listingBox, article.listing-card, li.listing",
    # Within a card:
    "price": ".price, .listingPrice, [class*=price]",
    "surface": ".surface, [class*=surface], [class*=area]",
    "rooms": ".rooms, [class*=room], [class*=pieces]",
    "location": ".location, [class*=location], [class*=city]",
    "link": "a",
    # Pagination: "next page" link selector
    "next_page": "a.next, a[rel=next]",
}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    ),
    "Accept-Language": "fr-TN,fr;q=0.9,en;q=0.8",
}

REQUEST_DELAY_SECONDS = 2.0  # be polite -- don't hammer the server
REQUEST_TIMEOUT = 15

# Keyword patterns to detect amenities from free-text (French + common variants)
AMENITY_PATTERNS = {
    "has_garden": r"\bjardin\b",
    "has_pool": r"\bpiscine\b",
    "has_garage": r"\bgarage\b|\bparking\b",
}

# Tunisian "S+N" room notation: S+1 = 1 bedroom, S+3 = salon + 3 bedrooms, etc.
ROOM_PATTERN = re.compile(r"S\s*\+\s*(\d+)", re.IGNORECASE)
SURFACE_PATTERN = re.compile(r"(\d[\d\s]*)\s*m[²2]", re.IGNORECASE)
PRICE_PATTERN = re.compile(r"(\d[\d\s.,]*)\s*(?:DT|TND|dinars?)?", re.IGNORECASE)

GOVERNORATES = [
    "Tunis", "Ariana", "Ben Arous", "Manouba", "Nabeul", "Zaghouan", "Bizerte",
    "Béja", "Jendouba", "Kef", "Siliana", "Sousse", "Monastir", "Mahdia",
    "Sfax", "Kairouan", "Kasserine", "Sidi Bouzid", "Gabès", "Medenine",
    "Tataouine", "Gafsa", "Tozeur", "Kebili",
]


@dataclass
class Listing:
    url: str
    price_tnd: Optional[float]
    surface_m2: Optional[float]
    rooms_s_plus: Optional[int]
    governorate: Optional[str]
    raw_location: Optional[str]
    has_garden: int
    has_pool: int
    has_garage: int
    raw_text: str


def fetch_page(url: str) -> Optional[BeautifulSoup]:
    """Fetch a page and return parsed soup, or None on failure."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
        return BeautifulSoup(resp.text, "lxml")
    except requests.RequestException as e:
        print(f"[WARN] Failed to fetch {url}: {e}", file=sys.stderr)
        return None


def fetch_with_playwright(url: str) -> Optional[BeautifulSoup]:
    """
    Fallback for JS-rendered sites. Requires:
        pip install playwright && playwright install chromium
    Swap this in for fetch_page() inside scrape() if requests returns 0 cards
    even after SELECTORS is correctly configured (i.e. the HTML truly isn't
    in the initial page load).
    """
    from playwright.sync_api import sync_playwright

    with sync_playwright() as p:
        browser = p.chromium.launch()
        page = browser.new_page(user_agent=HEADERS["User-Agent"])
        page.goto(url, timeout=30000)
        try:
            page.wait_for_selector(SELECTORS["card"], timeout=10000)
        except Exception:
            pass  # fall through -- parse whatever loaded, caller handles 0 cards
        html = page.content()
        browser.close()
    return BeautifulSoup(html, "lxml")


def extract_number(text: str, pattern: re.Pattern) -> Optional[float]:
    if not text:
        return None
    match = pattern.search(text)
    if not match:
        return None
    cleaned = match.group(1).replace(" ", "").replace(",", ".")
    try:
        return float(cleaned)
    except ValueError:
        return None


def extract_governorate(text: str) -> Optional[str]:
    if not text:
        return None
    for gov in GOVERNORATES:
        if gov.lower() in text.lower():
            return gov
    return None


def parse_card(card, base_url: str) -> Optional[Listing]:
    def text_of(selector: str) -> str:
        el = card.select_one(selector)
        return el.get_text(" ", strip=True) if el else ""

    link_el = card.select_one(SELECTORS["link"])
    href = link_el.get("href") if link_el else None
    full_url = urljoin(base_url, href) if href else base_url

    raw_text = card.get_text(" ", strip=True)
    price_text = text_of(SELECTORS["price"]) or raw_text
    surface_text = text_of(SELECTORS["surface"]) or raw_text
    rooms_text = text_of(SELECTORS["rooms"]) or raw_text
    location_text = text_of(SELECTORS["location"]) or raw_text

    price = extract_number(price_text, PRICE_PATTERN)
    surface = extract_number(surface_text, SURFACE_PATTERN)
    rooms_match = ROOM_PATTERN.search(rooms_text)
    rooms = int(rooms_match.group(1)) if rooms_match else None
    governorate = extract_governorate(location_text)

    lower_text = raw_text.lower()
    amenities = {
        key: int(bool(re.search(pat, lower_text)))
        for key, pat in AMENITY_PATTERNS.items()
    }

    # Skip cards that clearly aren't listings (ads, empty containers)
    if price is None and surface is None:
        return None

    return Listing(
        url=full_url,
        price_tnd=price,
        surface_m2=surface,
        rooms_s_plus=rooms,
        governorate=governorate,
        raw_location=location_text or None,
        raw_text=raw_text[:300],  # trimmed for sanity in the CSV
        **amenities,
    )


def scrape(start_url: str, max_pages: int = 1, use_playwright: bool = False) -> list:
    """Crawl up to max_pages pages starting at start_url, return list[Listing]."""
    listings: list = []
    current_url = start_url
    fetcher = fetch_with_playwright if use_playwright else fetch_page

    for page_num in range(1, max_pages + 1):
        print(f"[INFO] Fetching page {page_num}: {current_url}")
        soup = fetcher(current_url)
        if soup is None:
            break

        cards = soup.select(SELECTORS["card"])
        print(f"[INFO]   Found {len(cards)} candidate cards")
        if not cards:
            print(
                "[WARN] No cards matched SELECTORS['card']. Inspect the page "
                "HTML (right-click a listing -> Inspect) and update SELECTORS, "
                "or try use_playwright=True if the listings are JS-rendered."
            )
            break

        page_count = 0
        for card in cards:
            item = parse_card(card, current_url)
            if item:
                listings.append(item)
                page_count += 1
        print(f"[INFO]   Parsed {page_count} valid listings from this page")

        next_el = soup.select_one(SELECTORS["next_page"])
        next_href = next_el.get("href") if next_el else None
        if not next_href:
            print("[INFO] No next-page link found -- stopping.")
            break
        current_url = urljoin(current_url, next_href)

        if page_num < max_pages:
            time.sleep(REQUEST_DELAY_SECONDS)

    return listings


def save_csv(listings: list, out_path: str) -> None:
    if not listings:
        print("[WARN] Nothing to save.")
        return
    fieldnames = list(asdict(listings[0]).keys())
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for item in listings:
            writer.writerow(asdict(item))
    print(f"[INFO] Saved {len(listings)} listings to {out_path}")


def run_scrape(url: str, max_pages: int = 1, out_path: Optional[str] = None,
                use_playwright: bool = False):
    """
    Convenience wrapper for notebooks: scrapes, optionally saves CSV, and
    returns a pandas DataFrame directly -- no argparse involved, so this is
    safe to call from a Jupyter cell.

        from scrape_apartments import run_scrape
        df = run_scrape("https://www.mubawab.tn/fr/sc/appartements-a-vendre",
                         max_pages=5, out_path="mubawab_data.csv")
    """
    import pandas as pd

    listings = scrape(url, max_pages=max_pages, use_playwright=use_playwright)
    if out_path:
        save_csv(listings, out_path)
    return pd.DataFrame([asdict(item) for item in listings])


def main():
    # parse_known_args() ignores extra args (like Jupyter's -f <connection-file>)
    # instead of crashing, so this remains safe even if accidentally run with %run
    # in a kernel that injects its own flags.
    parser = argparse.ArgumentParser(description="Scrape apartment listings from a URL.")
    parser.add_argument("url", help="Listing search-results page URL")
    parser.add_argument("--pages", type=int, default=1, help="Max pages to crawl")
    parser.add_argument("--out", default="listings.csv", help="Output CSV path")
    parser.add_argument("--playwright", action="store_true",
                         help="Use Playwright (for JS-rendered sites) instead of requests")
    args, _unknown = parser.parse_known_args()

    listings = scrape(args.url, max_pages=args.pages, use_playwright=args.playwright)
    save_csv(listings, args.out)


if __name__ == "__main__":
    main()

[INFO] Fetching page 1: C:\Users\user\AppData\Roaming\jupyter\runtime\kernel-b23b075b-c8f5-460c-89a6-42bfab14dd5b.json
[WARN] Nothing to save.


[WARN] Failed to fetch C:\Users\user\AppData\Roaming\jupyter\runtime\kernel-b23b075b-c8f5-460c-89a6-42bfab14dd5b.json: No connection adapters were found for 'C:\\Users\\user\\AppData\\Roaming\\jupyter\\runtime\\kernel-b23b075b-c8f5-460c-89a6-42bfab14dd5b.json'


In [16]:
df = run_scrape(
    "https://www.mubawab.tn/fr/sc/appartements-a-vendre",
    max_pages=5,
    out_path="mubawab_data.csv"
)
df.describe()

[INFO] Fetching page 1: https://www.mubawab.tn/fr/sc/appartements-a-vendre


C:\Users\user\AppData\Local\Temp\ipykernel_2804\3889079788.py:115: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  return BeautifulSoup(resp.text, "lxml")


[INFO]   Found 34 candidate cards
[INFO]   Parsed 32 valid listings from this page
[INFO] No next-page link found -- stopping.
[INFO] Saved 32 listings to mubawab_data.csv


,price_tnd,surface_m2,rooms_s_plus,has_garden,has_pool,has_garage
count,10.00000,32.000000,3.000000,32.000000,32.000000,32.000000
mean,49.50000,276.468750,1.666667,0.125000,0.062500,0.781250
std,60.54062,863.174149,0.577350,0.336011,0.245935,0.420013
min,2.00000,4.000000,1.000000,0.000000,0.000000,0.000000
25%,3.00000,99.250000,1.500000,0.000000,0.000000,1.000000
50%,3.00000,120.500000,2.000000,0.000000,0.000000,1.000000
75%,118.75000,151.500000,2.000000,0.000000,0.000000,1.000000
max,124.00000,5000.000000,2.000000,1.000000,1.000000,1.000000
